# Génération des OMP avec surabondance et seuil

Compatibilité des cases surabondantes :
- deux cases sont compatibles si une au moins de leurs valeurs est commune

Compatibilité des colonnes :
- pas d'incompatibilité (*nbIncompatibles==0*)
- plus de valeurs compatibles que le seuil défini (*nbCompatibles>=seuilCompatibilite*)
    - seuil fixe à 10 (peut-être mieux 1% des lexèmes)

In [1]:
import pandas as pd
import numpy as np
import re,yaml,os
import itertools as it
import networkx as nx
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [2]:
pd.__version__

'2.2.3'

In [3]:
rep="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
seuilCompatibilite=10

In [4]:

def cleanCliques(lCliques):
    sCases=set(sum(lCliques,[]))
    for c in lCliques:
        wCliques.remove(c)
    for c in sCases:
        for w in wCliques:
            if c in w:
                w.remove(c)
    return
                

def addCliques(length):
    conflits=[]
    lCliques=[x for x in cliques if len(x)==length]
    if len(lCliques)>1:
        for (c1,c2) in it.combinations(lCliques, 2):
            inter=set(c1).intersection(set(c2))
            if inter:
                conflits.append([c1,c2])
    if conflits:
        ajouts=[]
        for c in lCliques:
            noConflit=True
            for conflit in conflits:
                if c in conflit:
                    noConflit=False
            if noConflit:
                ajouts.append(c)
        syncretismes.extend(ajouts)
        cleanCliques(ajouts)
        for c1,c2 in conflits:
            sC1=set(c1)
            sC2=set(c2)
            pivot=list(sC1.intersection(sC2))[0]
            dC1=sC1.difference(sC2)
            wC1=sum(g[pivot][c]["weight"] for c in dC1)
            dC2=sC2.difference(sC1)
            wC2=sum(g[pivot][c]["weight"] for c in dC2)
            # print (c1,wC1,c2,wC2)
    else:
        syncretismes.extend(lCliques)
        cleanCliques(lCliques)
    return

In [5]:
def canonExpansion(expansion):
    result=expansion.copy()
    if "fi2S" in result:
        if "fi3S" in result["fi2S"]:
            result[u"fi3S"]=result.pop("fi2S")
    if "ii1S" in result:
        if "ii3S" in result["ii1S"]:
            result[u"ii3S"]=result.pop("ii1S")
    if "pc2S" in result:
        if "pc3S" in result["pc2S"]:
            result[u"pc3S"]=result.pop("pc2S")
    if "ps2S" in result:
        if "ps3S" in result["ps2S"]:
            result[u"ps3S"]=result.pop("ps2S")        
    if "pi2S" in result:
        if "pi3S" in result["pi2S"]:
            result[u"pi3S"]=result.pop("pi2S")
    if "ppMP" in result:
        if "ppMS" in result["ppMP"]:
            result[u"ppMS"]=result.pop("ppMP")
    return result

In [6]:
def fusionFormes(row):
    result=[]
    for c in row:
        if c==c:
            result.extend(c.split(","))
    result=list(set(result))
    if result:
        result=",".join(result)
    else:
        result=np.NaN
    return result

In [7]:
inputFiles=["vlexique2-S%d.csv"%n for n in range(9)]
inputFiles+=["vlexique2-CV%d-%s%d.csv"%(n,t,i) for n in [2,3,5,10] for t in ["Train","Test"] for i in range(n)]
# inputFiles

for fParadigmes in inputFiles[:]:
    paradigmes=pd.read_csv(rep+fParadigmes,sep=";",encoding="utf8")
    cols=paradigmes.columns.tolist()
    cases=cols[:]
    cases.remove("lexeme")
    # print(fParadigmes,len(cases),", ".join(cases))

    g=nx.Graph()
    for (c1,c2) in it.combinations(cases, 2):
        # Traitement des cases sans surabondance
        c1SimpleVal=(paradigmes[c1].notnull())&(~paradigmes[c1].str.contains(",",na=False))
        c2SimpleVal=(paradigmes[c2].notnull())&(~paradigmes[c2].str.contains(",",na=False))
        
        incompatiblesSimples=paradigmes[c1SimpleVal & c2SimpleVal & (paradigmes[c1]!=paradigmes[c2])][[c1,c2]]
        compatiblesSimples=paradigmes[c1SimpleVal & c2SimpleVal & (paradigmes[c1]==paradigmes[c2])][[c1,c2]]
        
        nbIncompatibles=len(incompatiblesSimples)
        nbCompatibles=len(compatiblesSimples)
        # print(c1,c2,nbIncompatibles,nbCompatibles)

        # Traitement des cases avec surabondance
        c1MultipleVal=((paradigmes[c1].notnull())&(paradigmes[c2].notnull())&(paradigmes[c1].str.contains(",",na=False)))
        c2MultipleVal=((paradigmes[c1].notnull())&(paradigmes[c2].notnull())&(paradigmes[c2].str.contains(",",na=False)))
        multiples=paradigmes[c1MultipleVal|c2MultipleVal][[c1,c2]].apply(lambda x: x.str.split(',')).explode(c1).explode(c2)
        
        if len(multiples)>0:
            # si il y a des surabondances (multiples n'est pas vide)
            compatiblesMultiples=multiples[multiples[c1]==multiples[c2]][[c1,c2]]
            # on extrait les compatibilités
            nbCompatiblesMultiples=len(compatiblesMultiples.index.unique())
            # on compte les lexèmes compatibles
            if nbCompatiblesMultiples==len(multiples.index.unique()):
                # si le nombre de lexèmes compatibles est le même que le nombre de lexèmes avec surabondance
                nbCompatibles+=len(compatiblesMultiples)
                # on ajoute les compatibilités
            else:
                # sinon on ajoute les incompatibilités
                nbIncompatibles+=len(multiples.index.unique())-len(compatiblesMultiples.index.unique())
        
        # print(c1,c2,nbIncompatibles,nbCompatibles)
        
        if nbIncompatibles==0 and nbCompatibles>=seuilCompatibilite:
            # si il n'y a pas d'incompatibilité et qu'il y a suffisamment de compatibilités, les cases sont fusionnables
            # on ajoute le lien dans le graphe
            g.add_edge(c1,c2,weight=nbCompatibles)
    cliques=list(nx.find_cliques(g))
    # on extrait les cliques pour avoir les ensembles de cases fusionnables toutes deux à deux entre elles

    cliques=sorted(cliques, key=len, reverse=True)
    # print(cliques)
    maxLenClique=len(cliques[0])
    wCliques=(cliques[:])

    syncretismes=[]

    # on extrait les cliques en partant des plus grandes et en éliminant les éléments déjà cliqués de la suite du traitement
    for i in range(maxLenClique):
        if maxLenClique-i>1:
            # print(maxLenClique-i)
            addCliques(maxLenClique-i)
    
    # on calcule le dictionnaire de syncrétismes
    syncretiques=set()
    for l in syncretismes:
        # print(l)
        for c in l:
            # print(c)
            syncretiques.add(c)
    
    # on remplace les clés du dictionnaire par les formes canoniques (3ème personne si possible)
    dExpansion=canonExpansion({s[0]:s for s in syncretismes if len(s)>1})

    print()
    print(fParadigmes,len(cases))

    # on remplit l'OMP avec les colonnes non syncrétiques
    omps=paradigmes[[c for c in cols if c not in syncretiques]].copy()

    # on ajoute les colonnes syncrétiques en fusionnant leurs valeurs
    for s in dExpansion:
        print (s,dExpansion[s])
        omps[s]=paradigmes[dExpansion[s]].apply(fusionFormes,axis=1)
    print()
    print()

    # on écrit le dictionnaire des morphomes
    with open(rep+fParadigmes.replace(".csv","-omp.yaml"),"w") as outFile:
        yaml.safe_dump(dExpansion,outFile)

    # on écrit les paradigmes morphomiques
    omps.dropna(thresh=2).to_csv(rep+fParadigmes.replace(".csv","-omp.csv"),encoding="utf8",sep=";",index=None)


vlexique2-S0.csv 51
ps3S ['ps3S', 'ps2S', 'ps1S', 'ps3P']
ii3S ['ii3S', 'ii1S', 'ii3P', 'ii2S']
ai3S ['ai3S', 'is3S', 'ai2S']
is3P ['is3P', 'is2S', 'is1S']
ppFP ['ppFP', 'ppFS']
fi3P ['fi3P', 'fi1P']
ppMS ['ppMS', 'ppMP']
pi3S ['pi2S', 'pi3S']



vlexique2-S1.csv 51
ps3S ['ps3S', 'ps2S', 'ps1S', 'ps3P']
ii3S ['ii3S', 'ii1S', 'ii3P', 'ii2S']
ai3S ['ai3S', 'is3S', 'ai2S']
ppFP ['ppFP', 'ppFS']
ppMS ['ppMS', 'ppMP']
is1S ['is1S', 'is2S']
fi3S ['fi2S', 'fi3S']
pc3S ['pc2S', 'pc1S', 'fi1S', 'pc3S']
pi3S ['pi2S', 'pi3S']



vlexique2-S2.csv 51
ps3S ['ps3S', 'ps2S', 'ps1S', 'ps3P']
ii3S ['ii3S', 'ii1S', 'ii3P', 'ii2S']
ai3S ['ai3S', 'is3S', 'ai2S']
ppFP ['ppFP', 'ppFS']
ppMS ['ppMS', 'ppMP']
is1S ['is1S', 'is2S']
pc3S ['pc3S', 'pc1S']
fi3S ['fi2S', 'fi3S']
pi3S ['pi2S', 'pi3S']



vlexique2-S3.csv 51
pc3P ['pc3P', 'pc2S', 'pc1S', 'fi1S', 'pc3S']
is2S ['is2S', 'ps1S', 'ps2S', 'ps3P', 'ps3S']
ii3S ['ii3S', 'ii1S', 'ii3P', 'ii2S']
ppFP ['ppFP', 'ppFS']
ppMS ['ppMS', 'ppMP']
fi3S ['fi2S', 'fi3S'